自定义工具描述description
使用@tool装饰器以后，就是1个可以被大模型识别的1个工具了，没有在tool中使用description，则要求必须在函数中增加docstring,然后会将docstring的全部内容视为1个description。如果2个都写了，则优先使用description

In [4]:
from typing import Literal

from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic_core.core_schema import json_schema
from rich import print as rprint
@tool(description="获取具体城市的天气")
def get_weather(city:str="北京"):
#docstring中的描述有严格的要求，Args必须首字母大写，冒号必须是英文的，参数名称city，如果有描述，则函数声明的时候，指定的参数也必须说明其类型
#一旦参数设定了默认值，convert_to_openai_tool的输出结果中久没有required这个子元素了
    """
    查询城市的天气

    Args:
        city : 具体的城市名称，比如'北京'，'重庆'

    Returns:
        一串字符串，描述指定城市的天气情况

    """
    return f"{city}当天天气清理"
# #4 将函数绑定在模型上
# model_with_tools=model.bind_tools(
#     [get_weather]
# )
#5 调用模型
response=convert_to_openai_tool(get_weather)
rprint(response)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {'properties': {'city': {'default': '北京', 'type': 'string'}}, 'type': 'object'}
    }
}

自定义工具描述description
使用@tool装饰器后，需要设定参数parse_docstring=True，才能让docstring中的Args:生效,并且会加上google style的严格约束，必须换行和使用英文冒号，然后首字母必须大写。


In [8]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool(description="获取具体城市的天气",parse_docstring=True,response_format="content")
def get_weather(city:str="北京"):
#docstring中的描述有严格的要求，Args必须首字母大写，冒号必须是英文的，参数名称city，如果有描述，则函数声明的时候，指定的参数也必须说明其类型
#一旦参数设定了默认值，convert_to_openai_tool的输出结果中久没有required这个子元素了
    """
    查询城市的天气

    Args:
        city : 具体的城市名称，比如'北京'，'重庆'

    Returns:
        一串字符串，描述指定城市的天气情况

    """
    return f"{city}当天天气清理"
# #4 将函数绑定在模型上
# model_with_tools=model.bind_tools(
#     [get_weather]
# )
#5 调用模型
response=convert_to_openai_tool(get_weather)
rprint(response)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {
                'city': {
                    'default': '北京',
                    'description': "具体的城市名称，比如'北京'，'重庆'",
                    'type': 'string'
                }
            },
            'type': 'object'
        }
    }
}

对工具进行命名name_or_callable ，但是开发中不推荐使用这个参数修改命名，因为大家都习惯了函数名作为工具名称

In [9]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool(description="获取具体城市的天气",parse_docstring=True,response_format="content",name_or_callable="getWeather")
def get_weather(city:str="北京"):
#docstring中的描述有严格的要求，Args必须首字母大写，冒号必须是英文的，参数名称city，如果有描述，则函数声明的时候，指定的参数也必须说明其类型
#一旦参数设定了默认值，convert_to_openai_tool的输出结果中久没有required这个子元素了
    """
    查询城市的天气

    Args:
        city : 具体的城市名称，比如'北京'，'重庆'

    Returns:
        一串字符串，描述指定城市的天气情况

    """
    return f"{city}当天天气清理"
# #4 将函数绑定在模型上
# model_with_tools=model.bind_tools(
#     [get_weather]
# )
#5 调用模型
response=convert_to_openai_tool(get_weather)
rprint(response)

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {
                'city': {
                    'default': '北京',
                    'description': "具体的城市名称，比如'北京'，'重庆'",
                    'type': 'string'
                }
            },
            'type': 'object'
        }
    }
}

自定义args_schema
方式1 使用pydantic模型定义

In [18]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from typing import Literal

class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )

@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather():
#docstring中的描述有严格的要求，Args必须首字母大写，冒号必须是英文的，参数名称city，如果有描述，则函数声明的时候，指定的参数也必须说明其类型
#一旦参数设定了默认值，convert_to_openai_tool的输出结果中久没有required这个子元素了

    return "当天天气清理"
# #4 将函数绑定在模型上
# model_with_tools=model.bind_tools(
#     [get_weather]
# )
#5 调用模型
response=convert_to_openai_tool(get_weather)
rprint(response)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {
                'city': {
                    'default': '北京市',
                    'description': "具体的城市名称，例如'北京市'、'重庆市'、''天津市",
                    'maxLength': 10,
                    'minLength': 2,
                    'type': 'string'
                },
                'unit': {
                    'description': '描述气温的单位，必须是摄氏度，凯文等单位',
                    'enum': ['celsius', 'fahrenheit', 'kelvin'],
                    'type': 'string'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

自定义args_schema
方式2 使用Json Schema定义

In [24]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from typing import Literal

json_schema={
    'properties': {
        'city': {
            'default': '北京市',
            'description': "具体的城市名称，例如'北京市'、'重庆市'、'天津市'111",
            'maxLength': 10,
            'minLength': 2,
            'type': 'string'
        },
        'unit': {
            'description': '描述气温的单位，必须是摄氏度，凯文等单位111',
            'enum': ['celsius', 'fahrenheit', 'kelvin'],
            'type': 'string'
        }
    },
    'required': ['unit'],
    'type': 'object'
}

@tool(description="获取具体城市的天气",args_schema=json_schema)
def get_weather():
#docstring中的描述有严格的要求，Args必须首字母大写，冒号必须是英文的，参数名称city，如果有描述，则函数声明的时候，指定的参数也必须说明其类型
#一旦参数设定了默认值，convert_to_openai_tool的输出结果中久没有required这个子元素了

    return "当天天气清理"
# #4 将函数绑定在模型上
# model_with_tools=model.bind_tools(
#     [get_weather]
# )
#5 调用模型
response=convert_to_openai_tool(get_weather)
rprint(response)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {
                'city': {
                    'default': '北京市',
                    'description': "具体的城市名称，例如'北京市'、'重庆市'、'天津市'111",
                    'maxLength': 10,
                    'minLength': 2,
                    'type': 'string'
                },
                'unit': {
                    'description': '描述气温的单位，必须是摄氏度，凯文等单位111',
                    'enum': ['celsius', 'fahrenheit', 'kelvin'],
                    'type': 'string'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}